# Task 3: Correlation Between News Sentiment and Stock Returns

### Objective

Analyze whether financial news sentiment has a measurable relationship with stock market daily returns using Natural Language Processing (NLP) and correlation analysis.

## 1. Install NLTK Resources

In [36]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Hemen\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## 2. Import Required Libraries

In [37]:
import pandas as pd
import matplotlib.pyplot as plt

from nltk.sentiment import SentimentIntensityAnalyzer
from pandas.tseries.offsets import BDay
df = pd.read_csv("../data/raw/news.csv")
stock_df = pd.read_csv("../data/raw/AAPL.csv")

## 3. Initialize Sentiment Analyzer

### VADER Sentiment Model

### Why VADER?
VADER was selected because it performs well on short financial headlines and captures positive, neutral, and negative sentiment efficiently without requiring model training.

In [38]:
sia = SentimentIntensityAnalyzer()

## 4. Compute Sentiment Scores

### Generate Compound Sentiment Scores for Headlines

In [39]:
df["sentiment"] = df["headline"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

## 5. Handle Weekend News
### Align Weekend News to the Next Trading Day

Financial markets remain closed on weekends.
To improve alignment with stock prices, weekend news is shifted to the next business day.

In [40]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [41]:
df["aligned_date"] = df["date"]

df.loc[
    df["date"].dt.weekday >= 5,
    "aligned_date"
] = df["date"] + BDay(1)

## 6. Aggregate Daily Sentiment
Compute Average Daily Sentiment per Stock

In [42]:
df["aligned_date"] = pd.to_datetime(df["aligned_date"], errors="coerce")

daily_sentiment = (
    df.groupby(["stock", "aligned_date"], as_index=False)["sentiment"]
    .mean()
)

### Rename Date Column for Clarity

In [43]:
daily_sentiment.rename(
    columns={"aligned_date": "date"},
    inplace=True
)

In [44]:
merged = pd.merge(
    daily_sentiment,
    stock_df,
    left_on="date",
    right_on="Date"
)

ValueError: You are trying to merge on datetime64[us, UTC-04:00] and str columns for key 'date'. If you wish to proceed you should use pd.concat

## 7. Calculate Daily Stock Returns
### Compute Percentage Daily Returns

In [ ]:
stock_df["daily_return"] = (
    stock_df["Adj Close"].pct_change() * 100
)

### Convert Stock Dates to Datetime Format


In [ ]:
stock_df["Date"] = pd.to_datetime(
    stock_df["Date"]
).dt.date

## 8. Merge News and Stock Data
### Combine Sentiment Data with Price Data

In [ ]:
plt.scatter(
    merged["sentiment"],
    merged["daily_return"]
)

plt.xlabel("Sentiment")
plt.ylabel("Daily Return")
plt.title(f"Correlation: {correlation:.2f}")

plt.show()

## 9. Correlation Analysis
### Measure Relationship Between Sentiment and Returns

In [ ]:
correlation = merged["sentiment"].corr(
    merged["daily_return"]
)

print(
    f"Correlation between sentiment and returns: "
    f"{correlation:.4f}"
)

## 10. Data Visualization
### Scatter Plot: Sentiment vs Daily Return

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    merged["sentiment"],
    merged["daily_return"]
)

plt.xlabel("Sentiment Score")
plt.ylabel("Daily Return (%)")

plt.title(
    f"Sentiment vs Daily Return "
    f"(Correlation = {correlation:.2f})"
)

plt.grid(True)

plt.show()

## Sentiment Category Bar Chart

In [ ]:
def categorize_sentiment(score):
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    else:
        return "neutral"

merged["sentiment_category"] = merged["sentiment"].apply(
    categorize_sentiment
)

category_returns = (
    merged.groupby("sentiment_category")["daily_return"]
    .mean()
)

plt.figure(figsize=(6,4))

category_returns.plot(kind="bar")

plt.ylabel("Average Daily Return (%)")
plt.title("Average Daily Return by Sentiment Category")

plt.grid(axis="y")

plt.show()

NameError: name 'merged' is not defined

## Interpretation

The correlation between news sentiment and daily stock returns was weak and slightly positive. This suggests that positive news headlines may be associated with higher stock returns, but the relationship is not strong enough to indicate reliable predictive power.

Several limitations may affect the analysis. Stock prices are influenced by many external factors such as macroeconomic conditions, earnings reports, and investor behavior. Additionally, sentiment effects may appear with time lags rather than on the same trading day. The use of simple lexicon-based sentiment analysis may also fail to capture complex financial language and sarcasm in headlines.
